# News Intelligence — GPU Inference Server

Run this notebook in a Kaggle session with **GPU T4 × 2** (or P100/T4 × 1 with memory-mapped weights).

## What this does
1. Installs the gateway and all model dependencies
2. Downloads model weights from HuggingFace
3. Starts the inference gateway on port 9100
4. Exposes port 9100 via a Cloudflare Quick Tunnel
5. Prints the tunnel URL and the command to update Codespaces

## After starting
Copy the tunnel URL printed at the bottom, then run in Codespaces:
```bash
make set-llm-url URL=https://xxx.trycloudflare.com
```

## Session limits
Kaggle sessions run for up to 12 hours. When the session ends the tunnel dies and
processing stages will start retrying. Just start a new session and re-run this notebook.
No data is stored on Kaggle — everything lives in Codespaces (MinIO + Pixeltable).

## Cell 1 — secrets

Add `INFERENCE_API_KEY` as a Kaggle secret (the same value as in your Codespaces `.env`).
The cell reads it from the Kaggle secrets API so it is never visible in the notebook output.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
API_KEY = secrets.get_secret("INFERENCE_API_KEY")
assert len(API_KEY) >= 16, "INFERENCE_API_KEY must be >= 16 chars"
os.environ["INFERENCE_API_KEY"] = API_KEY
os.environ["INFERENCE_BACKEND"] = "gpu"

# Optional: override model ids if you want different checkpoints
# os.environ["MODEL_ASR"] = "openai/whisper-large-v3"
# os.environ["MODEL_VLM"] = "Qwen/Qwen2-VL-7B-Instruct"
# os.environ["MODEL_EMBED_TEXT"] = "BAAI/bge-m3"
# os.environ["MODEL_EMBED_IMAGE"] = "google/siglip-so400m-patch14-384"
# os.environ["MODEL_RERANK"] = "BAAI/bge-reranker-v2-m3"
print("Secrets loaded. API key ends with:", API_KEY[-4:])

## Cell 2 — install inference gateway and GPU dependencies

In [ ]:
# EasyOCR — downloads Arabic + English models on first use (~200 MB)
# gpu=False: leave CUDA for the transformer models
print("Downloading EasyOCR models …")
import easyocr
reader = easyocr.Reader(["en", "ar"], gpu=False)
# Quick smoke test
import numpy as np
dummy = np.zeros((64, 256, 3), dtype=np.uint8)
_ = reader.readtext(dummy)
del reader, _
print("EasyOCR OK")


## Cell 3 — install the inference gateway source

The gateway is in the `services/inference/` directory of the repo.
We install it from source so the GPU backend (`backends/gpu.py`) is available.

In [ ]:
%%bash
# Clone the repo (or update if already present)
if [ -d "/kaggle/working/learn_gpu" ]; then
    cd /kaggle/working/learn_gpu && git pull --ff-only
else
    git clone --depth 1 --branch feat/mvp https://github.com/Ajay263/learn_gpu.git /kaggle/working/learn_gpu
fi

pip install -q -e /kaggle/working/learn_gpu/services/inference
echo "Gateway installed: $(newsintel-inference --help 2>&1 | head -1)"

## Cell 4 — pre-download model weights

Downloading happens here (with visible progress) so the gateway starts instantly.
Total download: ~14 GB. Takes 5–10 min on Kaggle.

**Note:** After the first run Kaggle caches weights in `/root/.cache/huggingface/`.
Subsequent runs in the same session skip this step automatically.

In [ ]:
import os
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  {gb:.1f} GB")

MODEL_VLM = os.environ.get("MODEL_VLM", "Qwen/Qwen2-VL-7B-Instruct")
MODEL_EMBED_TEXT = os.environ.get("MODEL_EMBED_TEXT", "BAAI/bge-m3")
MODEL_EMBED_IMAGE = os.environ.get("MODEL_EMBED_IMAGE", "google/siglip-so400m-patch14-384")
MODEL_RERANK = os.environ.get("MODEL_RERANK", "BAAI/bge-reranker-v2-m3")

In [ ]:
# Whisper (openai-whisper downloads to ~/.cache/whisper)
print("Downloading Whisper large-v3 …")
import whisper
_ = whisper.load_model("large-v3", device="cpu")  # load to CPU first to verify download
del _
print("Whisper OK")

In [ ]:
# Qwen2-VL (vision-language model used for both captioning and chat)
print(f"Downloading {MODEL_VLM} …")
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_VLM, torch_dtype=torch.float16, quantization_config=bnb, device_map="auto"
)
proc = AutoProcessor.from_pretrained(MODEL_VLM)
print(f"{MODEL_VLM} OK — VRAM after load:")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {used:.1f}/{total:.1f} GB used")
del vlm, proc

In [ ]:
# BGE-M3 (text embeddings) and BGE-reranker-v2-m3
print(f"Downloading {MODEL_EMBED_TEXT} and {MODEL_RERANK} …")
from sentence_transformers import SentenceTransformer, CrossEncoder
_ = SentenceTransformer(MODEL_EMBED_TEXT, device="cuda")
__ = CrossEncoder(MODEL_RERANK, max_length=512, device="cuda")
del _, __
print("Embedder + reranker OK")

In [ ]:
# SigLIP (image embeddings)
print(f"Downloading {MODEL_EMBED_IMAGE} …")
from transformers import AutoProcessor, AutoModel
_ = AutoModel.from_pretrained(MODEL_EMBED_IMAGE, torch_dtype=torch.float16)
del _
print("SigLIP OK")

In [ ]:
# EasyOCR — downloads Arabic + English models on first use (~200 MB)
# gpu=False: leave CUDA for the transformer models
print("Downloading EasyOCR models …")
import easyocr
reader = easyocr.Reader(["en", "ar"], gpu=False)
# Quick smoke test
import numpy as np
dummy = np.zeros((64, 256, 3), dtype=np.uint8)
_ = reader.readtext(dummy)
del reader, _
print("EasyOCR OK")


## Cell 5 — start the gateway (background process)

In [ ]:
import subprocess, time, httpx, os

gateway_proc = subprocess.Popen(
    ["newsintel-inference", "--host", "0.0.0.0", "--port", "9100"],
    env={**os.environ},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for the gateway to start (models load lazily, so /livez is ready immediately)
for attempt in range(30):
    time.sleep(2)
    try:
        r = httpx.get("http://localhost:9100/livez", timeout=3)
        if r.status_code == 200:
            print("Gateway live after", (attempt + 1) * 2, "s")
            break
    except Exception:
        pass
else:
    out = gateway_proc.stdout.read(2000).decode(errors="replace")
    raise RuntimeError(f"Gateway did not start.\n{out}")

# Quick health check — models are lazy so this just confirms the server is up
health = httpx.get(
    "http://localhost:9100/v1/health",
    headers={"X-API-Key": os.environ["INFERENCE_API_KEY"]},
    timeout=10,
).json()
print("Backend:", health["backend"])
print("Status: ", health["status"])

## Cell 6 — expose via Cloudflare Quick Tunnel

In [ ]:
%%bash
# Download cloudflared if not present
if ! command -v cloudflared &>/dev/null; then
    echo "Downloading cloudflared …"
    curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -o /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
fi
echo "cloudflared $(cloudflared --version)"

In [ ]:
import subprocess, threading, re, time

tunnel_url = None
tunnel_log_lines = []

def _run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:9100"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for raw in proc.stdout:
        line = raw.decode(errors="replace").strip()
        tunnel_log_lines.append(line)
        m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if m:
            tunnel_url = m.group(0)

t = threading.Thread(target=_run_tunnel, daemon=True)
t.start()

# Wait up to 30 s for the tunnel URL
for _ in range(15):
    if tunnel_url:
        break
    time.sleep(2)

if not tunnel_url:
    print("Tunnel log (last 20 lines):")
    print("\n".join(tunnel_log_lines[-20:]))
    raise RuntimeError("Could not get tunnel URL — see log above")

print("=" * 60)
print("TUNNEL URL:", tunnel_url)
print("=" * 60)
print()
print("Run this in your Codespaces terminal to connect:")
print(f"  make set-llm-url URL={tunnel_url}")
print()
print("Or set manually:")
print(f"  echo 'AI_INFERENCE_BASE_URL={tunnel_url}' >> .env")
print(f"  make restart-app")

## Cell 7 — smoke test through the tunnel

Verifies the tunnel is working AND the API key is accepted from outside Kaggle.

In [ ]:
import httpx, os, time

HEADERS = {"X-API-Key": os.environ["INFERENCE_API_KEY"]}

# Retry a few times — tunnel may take a moment to propagate DNS
for attempt in range(6):
    try:
        r = httpx.get(f"{tunnel_url}/livez", timeout=10)
        assert r.status_code == 200
        print(f"livez OK (attempt {attempt+1})")
        break
    except Exception as e:
        print(f"attempt {attempt+1}: {e}")
        time.sleep(5)

# Health
h = httpx.get(f"{tunnel_url}/v1/health", headers=HEADERS, timeout=15).json()
print(f"health: {h['status']} backend={h['backend']}")

# Quick embed test (triggers lazy model load)
import json
print("Testing text embed (will trigger model load — may take 30-120 s) …")
r = httpx.post(
    f"{tunnel_url}/v1/embed/text",
    headers={**HEADERS, "Content-Type": "application/json"},
    content=json.dumps({"texts": ["hello world"], "purpose": "query"}),
    timeout=180,
)
assert r.status_code == 200, r.text
vec = r.json()
print(f"embed OK: dim={vec['dim']} model={vec['model']['name']}")
print()
print("All checks passed. Gateway is live.")
print(f"Copy this URL to Codespaces: {tunnel_url}")

## Cell 8 — keep alive (optional)

Run this cell to print a heartbeat every 10 minutes and confirm the tunnel is still up.
Stop it with the kernel interrupt button when done.

In [ ]:
import time, httpx, datetime, os

HEADERS = {"X-API-Key": os.environ["INFERENCE_API_KEY"]}

while True:
    try:
        r = httpx.get(f"{tunnel_url}/livez", timeout=5)
        status = "OK" if r.status_code == 200 else f"HTTP {r.status_code}"
    except Exception as e:
        status = f"ERROR: {e}"
    print(datetime.datetime.now().isoformat(timespec='seconds'), tunnel_url, status)
    time.sleep(600)